In [48]:
import pandas as pd
from sklearn.neighbors import NearestNeighbors
import matplotlib as plt
import numpy as np

In [49]:
df = pd.read_csv("../data/dbscan_clustered_parquet2.csv")
df.shape

(1229374, 442)

In [50]:
cluster_modes = df[df['cluster_labels'] != -1].groupby('cluster_labels').agg(pd.Series.mode) # gets the mode of every value, similar to a centriod of kmodes 
cluster_means = df[df['cluster_labels'] != -1].groupby('cluster_labels').mean() #gets the prevalence of a feature per cluster
cluster_amount = len(set(df[df['cluster_labels'] != -1]["cluster_labels"]))
df[df["cluster_labels"] != -1].shape
print(cluster_amount)

97


In [ ]:
cluster_modes.iloc[:,:5]

97


In [52]:
for cluster in cluster_modes.index:
    print(f"Cluster {cluster}: {df[df["cluster_labels"] == cluster].shape}")


Cluster 0: (32470, 442)
Cluster 1: (5633, 442)
Cluster 2: (439411, 442)
Cluster 3: (25222, 442)
Cluster 4: (2400, 442)
Cluster 5: (5568, 442)
Cluster 6: (3137, 442)
Cluster 7: (43301, 442)
Cluster 8: (1398, 442)
Cluster 9: (916, 442)
Cluster 10: (1787, 442)
Cluster 11: (3623, 442)
Cluster 12: (1389, 442)
Cluster 13: (1263, 442)
Cluster 14: (39547, 442)
Cluster 15: (94884, 442)
Cluster 16: (2434, 442)
Cluster 17: (2332, 442)
Cluster 18: (3789, 442)
Cluster 19: (164498, 442)
Cluster 20: (3695, 442)
Cluster 21: (2074, 442)
Cluster 22: (57313, 442)
Cluster 23: (5672, 442)
Cluster 24: (1121, 442)
Cluster 25: (10622, 442)
Cluster 26: (1361, 442)
Cluster 27: (7019, 442)
Cluster 28: (5511, 442)
Cluster 29: (2161, 442)
Cluster 30: (3974, 442)
Cluster 31: (1502, 442)
Cluster 32: (2354, 442)
Cluster 33: (899, 442)
Cluster 34: (1350, 442)
Cluster 35: (3101, 442)
Cluster 36: (2182, 442)
Cluster 37: (4832, 442)
Cluster 38: (1512, 442)
Cluster 39: (1050, 442)
Cluster 40: (1300, 442)
Cluster 41: (943,

In [53]:
discriminating_features = []

for col in cluster_modes.columns:
    temp = set()
    for cluster in cluster_modes[col]:
        temp.add(cluster)
        
    if len(temp) > 1:
        discriminating_features.append(col)
       
print("Discriminating Features: ") 
for feature in discriminating_features:
    print(f"\t{feature}")

Discriminating Features: 
	properties_images
	properties_notes
	properties_orientation_data
	properties_name
	geometry_type
	geometry_coordinates
	properties_samples
	properties_altitude_accuracy
	properties_lng
	properties_image_basemap
	properties_lat
	properties__3d_structures
	properties_trace_trace_feature
	properties_trace_trace_quality
	properties_trace_trace_type
	properties_trace_contact_type
	properties_trace_intrusive_contact_type
	properties_trace_trace_character
	properties_spot_radius
	properties_sed_character
	properties_sed_interval_interval_thickness
	properties_sed_interval_thickness_units
	properties_surface_feature_surface_feature_type
	properties_sed_lithologies
	properties_other_features
	properties_trace_depositional_contact_type
	properties_trace_tace_notes
	properties_trace_geologic_structure_type
	properties_trace_other_feature
	properties_trace_other_other_feature
	properties_trace_shear_sense
	properties_trace_other_contact_type
	properties_custom_fields_MIN

In [54]:
print(cluster_means[discriminating_features])

                properties_images  properties_notes  \
cluster_labels                                        
0                             1.0               1.0   
1                             1.0               1.0   
2                             0.0               0.0   
3                             1.0               0.0   
4                             1.0               1.0   
...                           ...               ...   
92                            0.0               0.0   
93                            0.0               1.0   
94                            0.0               0.0   
95                            0.0               0.0   
96                            0.0               0.0   

                properties_orientation_data  properties_name  geometry_type  \
cluster_labels                                                                
0                                       1.0              1.0            1.0   
1                                       0.0    

In [55]:
global_mean = df[df['cluster_labels'] != -1].drop(columns='cluster_labels').mean() # presence in the entire dataset
deviation = cluster_means[discriminating_features] / global_mean[discriminating_features]

deviation

,properties_images,properties_notes,properties_orientation_data,properties_name,geometry_type,geometry_coordinates,properties_samples,properties_altitude_accuracy,properties_lng,properties_image_basemap,...,properties_rock_unit_era,properties_rock_unit_period,properties_rock_unit_epoch,properties_rock_unit_group_unit_type,properties_custom_fields_rock_type,properties_custom_fields_rock_class,properties_custom_fields_TYPE,properties_custom_fields_gid,properties_custom_fields_state,properties_custom_fields_county
cluster_labels,,,,,,,,,,,,,,,,,,,,,
0,8.953454,5.075414,3.141734,1.001179,1.008998,1.008998,0.000000,0.000000,0.0,0.0,...,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
1,8.953454,5.075414,0.000000,1.001179,1.008998,1.008998,48.235606,0.000000,0.0,0.0,...,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
2,0.000000,0.000000,0.000000,1.001179,1.008998,1.008998,0.000000,0.000000,0.0,0.0,...,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
3,8.953454,0.000000,3.141734,1.001179,1.008998,1.008998,0.000000,0.000000,0.0,0.0,...,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
4,8.953454,5.075414,3.141734,1.001179,1.008998,1.008998,0.000000,62.411053,0.0,0.0,...,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
92,0.000000,0.000000,0.000000,1.001179,1.008998,1.008998,0.000000,0.000000,0.0,0.0,...,367.809933,367.809933,0.0,549.615128,0.0,0.0,0.0,0.0,0.0,0.0
93,0.000000,5.075414,3.141734,1.001179,1.008998,1.008998,0.000000,0.000000,0.0,0.0,...,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
94,0.000000,0.000000,0.000000,1.001179,1.008998,1.008998,0.000000,0.000000,0.0,0.0,...,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0


In [56]:
significant_deviations = {}

for cluster in deviation.index:
    features = deviation.loc[cluster]
    significant = features[features >= 2].sort_values(ascending=False)
    significant_deviations[cluster] = significant

for cluster, features in significant_deviations.items():
    print(f"\nCluster {cluster}:")
    if len(features) == 0:
        print("\tNo significant features")
    else:
        for feature, ratio in features.items():
            print(f"\t{feature}: {ratio:.2f}x more present in cluster")



Cluster 0:
	properties_images: 8.95x more present in cluster
	properties_notes: 5.08x more present in cluster
	properties_orientation_data: 3.14x more present in cluster

Cluster 1:
	properties_samples: 48.24x more present in cluster
	properties_images: 8.95x more present in cluster
	properties_notes: 5.08x more present in cluster

Cluster 2:
	No significant features

Cluster 3:
	properties_images: 8.95x more present in cluster
	properties_orientation_data: 3.14x more present in cluster

Cluster 4:
	properties_altitude_accuracy: 62.41x more present in cluster
	properties_images: 8.95x more present in cluster
	properties_notes: 5.08x more present in cluster
	properties_orientation_data: 3.14x more present in cluster

Cluster 5:
	properties_samples: 48.24x more present in cluster
	properties_images: 8.95x more present in cluster
	properties_notes: 5.08x more present in cluster
	properties_orientation_data: 3.14x more present in cluster

Cluster 6:
	properties_altitude_accuracy: 62.41x m